<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/roberta_anli_mini/blob/main/roberta_anli_mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install (Colab already has torch and transformers)
!pip -q install datasets scikit-learn

In [ ]:
# 2. Imports and settings
import random, math, numpy as np, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)            # should say: cuda

LABELS = ["entailment", "neutral", "contradiction"]   # label ids 0, 1, 2

# Mini settings. Your real run is in [brackets].
MODEL   = "roberta-base"
N_TRAIN = 2000    # examples per round   [all: 16,946 + 45,460 + 100,459]
N_DEV   = 300     # dev examples per round [all: 1,000 each]
MAX_LEN = 128     # [128]
BATCH   = 16      # [16]
ACCUM   = 2       # [2]  -> behaves like a batch of 32
EPOCHS  = 2       # [3]
LR      = 2e-5    # [1e-5]  a bit higher here because the mini run is short
WARMUP  = 0.06    # [6% of the updates]
WD      = 0.01    # [0.01]

device: cuda


In [ ]:
# 3. Data: load ANLI, look at one example, take a small slice of each round
ds = load_dataset("facebook/anli")
print(ds)                                   # splits: train_r1, dev_r1, test_r1, ... r2, r3

ex = ds["train_r1"][0]
print("\nPREMISE:   ", ex["premise"])
print("HYPOTHESIS:", ex["hypothesis"])
print("LABEL:     ", ex["label"], "=", LABELS[ex["label"]])

def take(split, n):
    return ds[split].shuffle(seed=SEED).select(range(n))

train_raw = concatenate_datasets([take(f"train_r{r}", N_TRAIN) for r in (1, 2, 3)]).shuffle(seed=SEED)
dev_raw   = concatenate_datasets([take(f"dev_r{r}", N_DEV) for r in (1, 2, 3)])
test_raw  = {r: ds[f"test_r{r}"] for r in (1, 2, 3)}     # full test sets, used ONCE at the end

print("\ntrain:", len(train_raw), "| dev:", len(dev_raw),
      "| test:", {f"R{r}": len(t) for r, t in test_raw.items()})

README.md:   0%|          | 0.00/7.98k [00:00<?, ?B/s]

plain_text/train_r1-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 3.14MB            

plain_text/train_r1-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

plain_text/dev_r1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  351kB            

plain_text/dev_r1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test_r2 split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_r3 split:   0%|          | 0/100459 [00:00<?, ? examples/s]

Generating dev_r3 split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test_r3 split:   0%|          | 0/1200 [00:00<?, ? examples/s]

DatasetDict({
    train_r1: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 16946
    })
    dev_r1: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 1000
    })
    test_r1: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 1000
    })
    train_r2: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 45460
    })
    dev_r2: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 1000
    })
    test_r2: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 1000
    })
    train_r3: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 100459
    })
    dev_r3: Dataset({
        features: ['uid', 'premise', 'hypothesis', 'label', 'reason'],
        num_rows: 12

In [ ]:
# 4. Tokenizer: see exactly what the model receives
tok = AutoTokenizer.from_pretrained(MODEL)

enc = tok(ex["premise"], ex["hypothesis"], truncation=True, max_length=MAX_LEN)
pieces = tok.convert_ids_to_tokens(enc["input_ids"])
print("number of tokens:", len(pieces))
print("first 20 pieces:", pieces[:20])
print("first 20 ids:   ", enc["input_ids"][:20])
print("where premise ends and hypothesis starts:", pieces[pieces.index("</s>") - 2 : pieces.index("</s>") + 4])
print("special ids:  <s> =", tok.cls_token_id, "  </s> =", tok.sep_token_id, "  <pad> =", tok.pad_token_id)
# 'Ġ' in a piece means 'there was a space before this word'

# How much do we cut off at MAX_LEN? (interview question: "what did truncation cost you?")
lengths = np.array([len(tok(p, h)["input_ids"])
                    for p, h in zip(train_raw["premise"][:500], train_raw["hypothesis"][:500])])
print(f"\npairs longer than {MAX_LEN} tokens: {(lengths > MAX_LEN).mean():.0%}  (median length {int(np.median(lengths))})")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

number of tokens: 80
first 20 pieces: ['<s>', 'The', 'ĠP', 'arma', 'Ġt', 'rolley', 'bus', 'Ġsystem', 'Ġ(', 'Italian', ':', 'Ġ"', 'Ret', 'e', 'Ġfil', 'ovi', 'aria', 'Ġdi', 'ĠP', 'arma']
first 20 ids:    [0, 133, 221, 15277, 326, 30974, 18924, 467, 36, 39251, 35, 22, 27814, 242, 14242, 19277, 8954, 2269, 221, 15277]
where premise ends and hypothesis starts: ['Ġroutes', '.', '</s>', '</s>', 'The', 'Ġt']
special ids:  <s> = 0   </s> = 2   <pad> = 1

pairs longer than 128 tokens: 5%  (median length 89)


In [ ]:
# 5. Tokenize everything and build batches
def encode(batch):
    return tok(batch["premise"], batch["hypothesis"], truncation=True, max_length=MAX_LEN)

def prepare(d):
    d = d.filter(lambda x: x["label"] != -1)                  # drop rows without a label
    drop = [c for c in d.column_names if c != "label"]       # keep only the label + tokens
    return d.map(encode, batched=True, remove_columns=drop)

collate = DataCollatorWithPadding(tok)   # pads each batch only to ITS longest pair
train_dl = DataLoader(prepare(train_raw), batch_size=BATCH, shuffle=True, collate_fn=collate)
dev_dl   = DataLoader(prepare(dev_raw), batch_size=64, collate_fn=collate)
test_dl  = {r: DataLoader(prepare(t), batch_size=64, collate_fn=collate) for r, t in test_raw.items()}

b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()})
# input_ids (16, N): 16 pairs, padded to the longest one in this batch
# labels (16,): one number per pair, 0/1/2

Filter:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

{'input_ids': (16, 128), 'attention_mask': (16, 128), 'labels': (16,)}


In [ ]:
# 6. Model: pretrained encoder + a NEW 3-way head
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3).to(device)
# The warning "Some weights ... newly initialized: classifier..." is EXPECTED: that's the new head.
print(model.classifier)      # dense 768->768, tanh, out_proj 768->3
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

def show(premise, hypothesis):
    model.eval()
    enc = tok(premise, hypothesis, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = logits.softmax(-1)
    print(f"logits {[round(x, 2) for x in logits.tolist()]}  ->  "
          + "  ".join(f"{l} {p:.2f}" for l, p in zip(LABELS, probs.tolist()))
          + f"   => {LABELS[int(probs.argmax())]}")

show("Nobody in the room spoke English.", "At least one person could speak English.")
# Before training the head is random, so the probabilities are close to 0.33 / 0.33 / 0.33

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=3, bias=True)
)
parameters: 125M
logits [-0.07, 0.04, -0.18]  ->  entailment 0.33  neutral 0.37  contradiction 0.30   => neutral


In [ ]:
# 7. Evaluation helpers
@torch.no_grad()
def predict(loader):
    model.eval()
    preds, gold = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        preds += logits.argmax(-1).tolist()
        gold  += batch["labels"].tolist()
    return np.array(preds), np.array(gold)

def accuracy(loader):
    p, g = predict(loader)
    return (p == g).mean()

In [ ]:
# 8. Train: the loop Trainer runs for you, written out by hand
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
updates_per_epoch = math.ceil(len(train_dl) / ACCUM)
total_updates = updates_per_epoch * EPOCHS
warmup_updates = int(WARMUP * total_updates)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_updates, total_updates)
print(f"{len(train_dl)} batches per epoch | {total_updates} weight updates | first {warmup_updates} are warmup")

best_acc, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_dl, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch["input_ids"],
                       attention_mask=batch["attention_mask"]).logits   # forward: 3 scores per pair
        loss = F.cross_entropy(logits, batch["labels"])                 # how wrong were we?
        (loss / ACCUM).backward()                                       # gradients, added up over ACCUM batches
        running += loss.item()

        if step % ACCUM == 0 or step == len(train_dl):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    # cap the gradient size
            optimizer.step()                                           # change the weights a tiny bit
            scheduler.step()                                           # warmup, then linear decay of LR
            optimizer.zero_grad()

        if step % 100 == 0:
            print(f"epoch {epoch}  step {step:4d}/{len(train_dl)}  avg loss {running / 100:.3f}  "
                  f"lr {scheduler.get_last_lr()[0]:.2e}")
            running = 0.0

    dev_acc = accuracy(dev_dl)
    print(f"== end of epoch {epoch}: dev accuracy {dev_acc:.3f}")
    if dev_acc > best_acc:                                             # keep the best checkpoint (by DEV)
        best_acc = dev_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("best dev accuracy:", round(float(best_acc), 3))
# Loss starts near ln(3) = 1.10 (random guessing between 3 classes). Watch whether it goes below that.

375 batches per epoch | 376 weight updates | first 22 are warmup
epoch 1  step  100/375  avg loss 1.087  lr 1.84e-05
epoch 1  step  200/375  avg loss 1.078  lr 1.56e-05
epoch 1  step  300/375  avg loss 1.073  lr 1.28e-05
== end of epoch 1: dev accuracy 0.319
epoch 2  step  100/375  avg loss 0.789  lr 7.80e-06
epoch 2  step  200/375  avg loss 0.724  lr 4.97e-06
epoch 2  step  300/375  avg loss 0.672  lr 2.15e-06
== end of epoch 2: dev accuracy 0.347
best dev accuracy: 0.347


In [ ]:
# 9. Test: each round separately, only once, at the end
for r, dl in test_dl.items():
    p, g = predict(dl)
    print(f"ANLI R{r} test accuracy: {(p == g).mean():.3f}")

p, g = predict(test_dl[1])
print("\nR1 per-class results:\n", classification_report(g, p, target_names=LABELS, digits=3))
print("R1 confusion matrix (rows = true label, columns = predicted):")
print(confusion_matrix(g, p))

ANLI R1 test accuracy: 0.417
ANLI R2 test accuracy: 0.356
ANLI R3 test accuracy: 0.350

R1 per-class results:
                precision    recall  f1-score   support

   entailment      0.371     0.476     0.417       334
      neutral      0.532     0.477     0.503       333
contradiction      0.363     0.297     0.327       333

     accuracy                          0.417      1000
    macro avg      0.422     0.417     0.416      1000
 weighted avg      0.422     0.417     0.416      1000

R1 confusion matrix (rows = true label, columns = predicted):
[[159  71 104]
 [104 159  70]
 [165  69  99]]


In [ ]:
# 10. Try your own pairs
show("Nobody in the room spoke English.", "At least one person could speak English.")
show("A man is playing a guitar on stage.", "A person is making music.")
show("The meeting was moved to Friday.", "The meeting is on Monday.")
show("Anna bought a red car in 2019.", "Anna likes the colour red.")

logits [0.81, -1.22, -0.09]  ->  entailment 0.65  neutral 0.09  contradiction 0.26   => entailment
logits [0.49, -1.69, 0.47]  ->  entailment 0.48  neutral 0.05  contradiction 0.47   => entailment
logits [-0.95, -0.46, 0.83]  ->  entailment 0.12  neutral 0.19  contradiction 0.69   => contradiction
logits [-0.57, 2.21, -1.2]  ->  entailment 0.06  neutral 0.91  contradiction 0.03   => neutral


## What to expect

This mini run sees **6,000** training pairs for 2 epochs. Your real run saw **162,865** pairs for 3 epochs (10.6 hours) and got **60.2 / 44.3 / 43.8**.

So here, expect roughly **R1 in the high 30s to mid 40s** and **R2/R3 close to chance (33%)**. That is normal: ANLI is built to be hard, and data size matters a lot.

## Experiments: each one answers an interview question

Change one setting, then re-run from **cell 6** (the model must start fresh each time).

1. **"Why 1e-5 and not 1e-3?"** Set `LR = 1e-3`. The loss stops falling and accuracy drops to about 33%: the model often predicts one class for everything. A big learning rate wrecks the pretrained weights.
2. **"Why train on all rounds?" (your ablation)** In cell 3, use `train_raw = take("train_r1", 6000)`, re-run cells 3–9. Compare R2/R3 with the all-rounds run.
3. **"What does truncation cost?"** Set `MAX_LEN = 64`, re-run cells 4–9. Cell 4 shows how many pairs get cut.
4. **"Why warmup?"** Set `WARMUP = 0` and `LR = 5e-5`, re-run from cell 6. Watch the first loss values.